# 03 — XGBoost Model Training + SHAP Explainability

Train an XGBoost regression model to predict log10(plastic emission) at river
outfalls, using the feature matrix from notebook 02.

Steps:
1. Load and prepare feature matrix
2. Define feature set and handle missing values
3. Train/validation split (stratified by continent)
4. XGBoost baseline + Optuna hyperparameter tuning
5. Evaluate on held-out set (R², RMSE, Spearman ρ)
6. SHAP feature importance and explainability
7. Save model and predictions

In [17]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from scipy import stats
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error

sys.path.insert(0, str(Path.cwd().parent / "src"))

DATA_PROC = Path("../data/processed")
RESULTS = Path("../results")

## 1. Load feature matrix

In [18]:
features = pd.read_csv(DATA_PROC / "feature_matrix_v1.csv")
print(f"Feature matrix: {features.shape}")
features.head()

Task was destroyed but it is pending!
task: <Task pending name='Task-148' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/homebrew/anaconda3/envs/riverplastic/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-149' coro=<Kernel.shell_main() running at /opt/homebrew/anaconda3/envs/riverplastic/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/homebrew/anaconda3/envs/riverplastic/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/homebrew/anaconda3/envs/riverplastic/lib/python3.11/site-packages/pandas/core/internals/blocks.py:650: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  return type(self)(values, placement=self._mgr_locs, ndim=self.ndim, refs=refs)
Task was destroyed but it is pending!
task: <Task pending name='Task-149' coro=<Kernel.shell_main() running at /opt/homebrew/anaconda3/envs/riverplastic/l

Feature matrix: (31819, 42)


,emission_ton,log_emission,lon,lat,DIS_AV_CMS,UPLAND_SKM,CATCH_SKM,ORD_STRA,LENGTH_KM,ENDORHEIC,...,plastic_pct,mpw_kg_cap_day,log_mpw_rate,runoff_mean_mm_yr,runoff_cv,runoff_anomaly,channel_width_m,elevation_m,slope_pct,nightlight_intensity
0,0.164904,-0.782769,168.797917,-46.580833,0.563,20.2,9.74,2.0,3.84,0.0,...,8.32,0.08029,-1.095341,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
1,0.124932,-0.903326,168.348750,-46.447083,0.394,15.7,15.83,1.0,5.41,0.0,...,8.32,0.08029,-1.095341,355.80830,0.704722,0.054035,NaN,NaN,NaN,1.393694
2,1.213370,0.083993,168.337083,-46.418750,5.445,222.5,0.44,3.0,0.56,0.0,...,8.32,0.08029,-1.095341,355.80830,0.704722,0.054035,NaN,NaN,NaN,11.979567
3,0.121138,-0.916720,168.021250,-46.357917,37.910,1549.4,4.30,4.0,1.45,0.0,...,8.32,0.08029,-1.095341,NaN,NaN,NaN,NaN,NaN,NaN,0.898780
4,0.197533,-0.704360,169.811250,-46.343750,599.733,20630.2,27.14,6.0,10.67,0.0,...,8.32,0.08029,-1.095341,108.02395,0.766468,-0.325201,NaN,NaN,NaN,0.000000


## 2. Define feature set and handle missing values

We use only features with >50% coverage. Missing values are filled with
column medians (XGBoost can handle NaN natively, but we fill for consistency
with SHAP and to avoid dropping rows with partial data).

In [19]:
# Define model features (drop target, spatial, categorical, and fully-missing columns)
drop_cols = [
    "emission_ton", "log_emission", "lon", "lat",
    "country_iso", "continent",
    "channel_width_m", "elevation_m", "slope_pct", "road_density_km_km2",
]

feature_cols = [c for c in features.columns if c not in drop_cols]
print(f"Model features ({len(feature_cols)}):")
for c in feature_cols:
    print(f"  {c}")

X = features[feature_cols].copy()
y = features["log_emission"].copy()

Model features (33):
  DIS_AV_CMS
  UPLAND_SKM
  CATCH_SKM
  ORD_STRA
  LENGTH_KM
  ENDORHEIC
  log_discharge
  log_upstream_area
  log_catch_area
  precip_mm_01
  precip_mm_02
  precip_mm_03
  precip_mm_04
  precip_mm_05
  precip_mm_06
  precip_mm_07
  precip_mm_08
  precip_mm_09
  precip_mm_10
  precip_mm_11
  precip_mm_12
  precip_annual_mm
  precip_max_month_mm
  precip_cv
  waste_kg_cap_day
  mismanaged_pct
  plastic_pct
  mpw_kg_cap_day
  log_mpw_rate
  runoff_mean_mm_yr
  runoff_cv
  runoff_anomaly
  nightlight_intensity


In [ ]:
# Check coverage
coverage = X.notna().mean().sort_values()
print("Feature coverage:")
for c, v in coverage.items():
    status = "OK" if v > 0.5 else "LOW — will be filled with median"
    print(f"  {c:30s} {v:.1%}  {status}")

In [ ]:
# Fill missing values with column medians
X_filled = X.fillna(X.median())
print(f"After fill — remaining NaN: {X_filled.isna().sum().sum()}")

## 3. Train/validation split

We hold out 20% of rivers for final evaluation, stratified by continent
to ensure geographic representation. The split uses the country as a group
key so rivers from the same country stay together.

In [ ]:
from sklearn.model_selection import train_test_split

# Use continent for stratification (fill NaN with 'Unknown')
strat_col = features["continent"].fillna("Unknown")

X_train, X_val, y_train, y_val = train_test_split(
    X_filled, y,
    test_size=0.2,
    random_state=42,
stratify=strat_col,
)

print(f"Train: {len(X_train):,} rivers")
print(f"Val:   {len(X_val):,} rivers")
print(f"\nTrain emission range: {10**y_train.min():.2f} – {10**y_train.max():.0f} ton/yr")
print(f"Val   emission range: {10**y_val.min():.2f} – {10**y_val.max():.0f} ton/yr")

## 4. XGBoost baseline model

In [ ]:
baseline_params = {
    "objective": "reg:squarederror",
    "max_depth": 6,
    "learning_rate": 0.1,
    "n_estimators": 300,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
}

model_baseline = xgb.XGBRegressor(**baseline_params)
model_baseline.fit(X_train, y_train)

y_pred_baseline = model_baseline.predict(X_val)
r2_base = r2_score(y_val, y_pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_val, y_pred_baseline))
spearman_base = stats.spearmanr(y_val, y_pred_baseline).statistic

print("=== Baseline XGBoost ===")
print(f"R²:         {r2_base:.4f}")
print(f"RMSE:       {rmse_base:.4f} (log10 ton/yr)")
print(f"Spearman ρ: {spearman_base:.4f}")

## 5. Optuna hyperparameter tuning

In [ ]:
import optuna

def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "random_state": 42,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return r2_score(y_val, y_pred)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest R²: {study.best_value:.4f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Train final model with best params
best_params = {**study.best_params, "objective": "reg:squarederror", "random_state": 42}
model_best = xgb.XGBRegressor(**best_params)
model_best.fit(X_train, y_train)

y_pred_best = model_best.predict(X_val)
r2_best = r2_score(y_val, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_val, y_pred_best))
spearman_best = stats.spearmanr(y_val, y_pred_best).statistic

print("=== Tuned XGBoost ===")
print(f"R²:         {r2_best:.4f}")
print(f"RMSE:       {rmse_best:.4f} (log10 ton/yr)")
print(f"Spearman ρ: {spearman_best:.4f}")
print()
print(f"Improvement over baseline: R² +{r2_best - r2_base:.4f}, Spearman +{spearman_best - spearman_base:.4f}")

## 6. Residual analysis

In [ ]:
residuals = y_val - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Actual vs predicted
axes[0].scatter(y_val, y_pred_best, alpha=0.3, s=5)
axes[0].plot([-3, 5], [-3, 5], "r--", lw=1)
axes[0].set_xlabel("Actual log10(ton/yr)")
axes[0].set_ylabel("Predicted log10(ton/yr)")
axes[0].set_title(f"Actual vs Predicted (R²={r2_best:.3f})")

# Residual distribution
axes[1].hist(residuals, bins=80, edgecolor="black", alpha=0.7)
axes[1].axvline(0, color="r", ls="--")
axes[1].set_xlabel("Residual (actual - predicted)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual distribution")

# Residual vs predicted
axes[2].scatter(y_pred_best, residuals, alpha=0.3, s=5)
axes[2].axhline(0, color="r", ls="--")
axes[2].set_xlabel("Predicted log10(ton/yr)")
axes[2].set_ylabel("Residual")
axes[2].set_title("Residual vs Predicted")

plt.tight_layout()
plt.savefig(RESULTS / "figures" / "03_residual_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. SHAP feature importance

In [ ]:
explainer = shap.TreeExplainer(model_best)
shap_values = explainer.shap_values(X_val)

# Global feature importance (mean |SHAP|)
shap_importance = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print("SHAP Feature Importance (top 15):")
print(shap_importance.head(15).to_string(index=False))

In [ ]:
# SHAP summary plot
shap.summary_plot(shap_values, X_val, max_display=15, show=False)
plt.savefig(RESULTS / "figures" / "03_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# SHAP bar plot
shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=15, show=False)
plt.savefig(RESULTS / "figures" / "03_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Compare with Meijer 2021 ranking

Generate updated predictions for all rivers and compare the ranking
with the original Meijer 2021 ranking.

In [ ]:
# Predict on all rivers
X_all = X_filled.copy()
y_pred_all = model_best.predict(X_all)
features["pred_log_emission"] = y_pred_all
features["pred_emission_ton"] = 10 ** y_pred_all

# Updated ranking
features["rank_updated"] = features["pred_emission_ton"].rank(ascending=False, method="min").astype(int)
features["rank_meijer"] = features["emission_ton"].rank(ascending=False, method="min").astype(int)
features["delta_rank"] = features["rank_meijer"] - features["rank_updated"]

# Top 10 comparison
print("Top 10 rivers — Updated model vs Meijer 2021:")
top10 = features.nsmallest(10, "rank_updated")
print(top10[["pred_emission_ton", "emission_ton", "lon", "lat", "rank_updated", "rank_meijer", "delta_rank"]].to_string())

In [ ]:
# Rank correlation between the two rankings
rank_corr = stats.spearmanr(features["rank_meijer"], features["rank_updated"]).statistic
print(f"Spearman rank correlation (Meijer vs Updated): {rank_corr:.4f}")

# How many top-1000 rivers are the same?
meijer_top1000 = set(features.nsmallest(1000, "rank_meijer").index)
updated_top1000 = set(features.nsmallest(1000, "rank_updated").index)
overlap = len(meijer_top1000 & updated_top1000)
print(f"Top-1000 overlap: {overlap}/1000 ({overlap/10:.1f}%)")

In [ ]:
# Biggest rank changes
print("Biggest risers (updated rank much higher than Meijer):")
risers = features.nlargest(10, "delta_rank")
print(risers[["pred_emission_ton", "emission_ton", "lon", "lat", "rank_updated", "rank_meijer", "delta_rank"]].to_string())

print("\nBiggest fallers (updated rank much lower than Meijer):")
fallers = features.nsmallest(10, "delta_rank")
print(fallers[["pred_emission_ton", "emission_ton", "lon", "lat", "rank_updated", "rank_meijer", "delta_rank"]].to_string())

## 9. Save model and predictions

In [ ]:
import joblib

# Save model
joblib.dump(model_best, DATA_PROC / "xgboost_model_v1.pkl")
print(f"Model saved to {DATA_PROC / 'xgboost_model_v1.pkl'}")

# Save predictions
output_cols = [
    "emission_ton", "pred_emission_ton", "log_emission", "pred_log_emission",
    "lon", "lat", "rank_meijer", "rank_updated", "delta_rank",
]
features[output_cols].to_csv(DATA_PROC / "updated_ranking_v1.csv", index=False)
print(f"Predictions saved to {DATA_PROC / 'updated_ranking_v1.csv'}")